# PP-OCRv6_small_det 파인튜닝 (Colab GPU)

런타임을 GPU로: `런타임 > 런타임 유형 변경 > T4 GPU`.

rec 파인튜닝 때와 같은 패턴(paddle 버전 고정+드라이브 wheel캐싱, --depth 1 클론).
3번 셀에서 로컬의 `data/generated/det_overlay_colab.zip` 업로드 필요.

v5_mobile_det(PPLCNetV3+RSEFPN) 대신 v6_small_det(PPLCNetV4+RepLKFPN) 사용 —
RepLKFPN이 넓은 dilated conv로 receptive field가 넓어 얇은 GD&T/거칠기 기호처럼
작은 심볼을 잡는 데 더 유리할 것으로 판단해 전환.

데이터: 실도면 배경 그대로 + 기존 텍스트박스 40%만 '글자별 왜곡+합성'으로 갈아끼움
(폰트 통짜합성의 '너무 완벽함' 문제 회피). 원본1 + 증강2 = 3배.

In [ ]:
# 1. 드라이브 마운트 + 의존성 설치 (paddlepaddle-gpu는 드라이브에 캐싱 — 최초 1회만 느림)
from google.colab import drive
drive.mount('/content/drive')

import glob, os
WHEEL_DIR = '/content/drive/MyDrive/paddle_wheels'
os.makedirs(WHEEL_DIR, exist_ok=True)
existing = glob.glob(f'{WHEEL_DIR}/paddlepaddle_gpu-3.3.1*.whl')
if existing:
    print(f'캐시 사용: {existing[0]}')
    !pip install "{existing[0]}" -q
else:
    print('캐시 없음 - 최초 1회 다운로드 (다음부턴 훨씬 빨라짐)')
    !pip download --timeout 300 --retries 5 paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ -d {WHEEL_DIR} --no-deps
    existing = glob.glob(f'{WHEEL_DIR}/paddlepaddle_gpu-3.3.1*.whl')
    !pip install "{existing[0]}" -q

!pip install lmdb scikit-image albumentations opencv-python pyclipper shapely rapidfuzz -q

In [ ]:
# 2. PaddleOCR 클론(--depth 1) + det 사전학습 가중치 다운로드
# rec_pretrained가 아니라 det_pretrained(전체 det모델, backbone전용 아님)
!git clone -q --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!mkdir -p pretrain_weights
!wget -q https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_det_pretrained.pdparams -O pretrain_weights/PP-OCRv6_small_det_pretrained.pdparams
print('done')

In [ ]:
# 3. 학습데이터 업로드 (로컬의 data/generated/det_overlay_colab.zip 선택)
# 실도면 배경 위에 40% 박스를 '글자별 왜곡+합성'으로 갈아끼운 det 데이터 (원본+증강 3배).
# ※ zip이 ~100MB라 files.upload()가 느리면, 드라이브에 직접 올린 뒤
#    !cp /content/drive/MyDrive/det_overlay_colab.zip . 로 대체 가능.
from google.colab import files
uploaded = files.upload()
!mkdir -p train_data_det
!unzip -q det_overlay_colab.zip -d train_data_det
!wc -l train_data_det/train_list.txt train_data_det/val_list.txt

In [ ]:
# 4. 파인튜닝 config 작성 (로컬에서 만든 것과 동일 — GPU용으로 use_gpu:true만 유지)
# 회전 rotate:[-50,50] — 사진 촬영 시 임의각도 강건성(det는 방향 무관, 박스는 IaaAugment가 자동 변환).
config_yaml = '''
Global:
  model_name: PP-OCRv6_small_det
  debug: false
  use_gpu: true
  use_ema: true
  ema_decay: 0.9997
  ema_decay_type: threshold
  epoch_num: &epoch_num 30
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/PP-OCRv6_small_det_finetune
  save_epoch_step: 5
  eval_batch_step:
  - 0
  - 200
  cal_metric_during_train: false
  checkpoints:
  pretrained_model: ./pretrain_weights/PP-OCRv6_small_det_pretrained
  save_inference_dir: null
  use_visualdl: false
  infer_img: doc/imgs_en/img_10.jpg
  save_res_path: ./output/det_db/predicts_db.txt
  d2s_train_image_shape: [3, 640, 640]
  distributed: true

Architecture:
  model_type: det
  algorithm: DB
  Transform: null
  Backbone:
    name: PPLCNetV4
    det: true
    model_size: small
  Neck:
    name: RepLKFPN
    out_channels: 96
    dilated_kernel_size: 7
    shortcut: true
  Head:
    name: DBHead
    k: 50
    fix_nan: true
    aux_in_channels: 96

Loss:
  name: DBLoss
  main_loss_type: DiceFocalLoss
  alpha: 5
  beta: 10
  focal_alpha: 0.25
  focal_gamma: 2.5
  aux_weight_p4: 0.2
  aux_weight_p3: 0.3
  aux_weight_p2: 0.4

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.001
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 1.0e-05

PostProcess:
  name: DBPostProcess
  thresh: 0.2
  box_thresh: 0.45
  max_candidates: 3000
  unclip_ratio: 1.4

Metric:
  name: DetMetric
  main_indicator: hmean

Train:
  dataset:
    name: SimpleDataSet
    data_dir: ./train_data_det/
    label_file_list:
      - ./train_data_det/train_list.txt
    ratio_list: [1.0]
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - DetLabelEncode: null
    - CopyPaste: null
    - IaaAugment:
        augmenter_args:
        - type: Fliplr
          args:
            p: 0.5
        - type: Affine
          args:
            p: 0.5
            rotate:
            - -50
            - 50
            fit_output: true
        - type: Resize
          args:
            size:
            - 0.5
            - 3
    - RandomCrop:
        size:
        - 640
        - 640
        max_tries: 50
        keep_ratio: true
    - MakeBorderMap:
        shrink_ratio: 0.4
        thresh_min: 0.3
        thresh_max: 0.7
        total_epoch: *epoch_num
    - MakeShrinkMap:
        shrink_ratio: 0.4
        min_text_size: 8
        total_epoch: *epoch_num
    - NormalizeImage:
        scale: 1./255.
        mean:
        - 0.485
        - 0.456
        - 0.406
        std:
        - 0.229
        - 0.224
        - 0.225
        order: hwc
    - ToCHWImage: null
    - KeepKeys:
        keep_keys:
        - image
        - threshold_map
        - threshold_mask
        - shrink_map
        - shrink_mask
  loader:
    shuffle: true
    drop_last: false
    batch_size_per_card: 8
    num_workers: 8

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./train_data_det/
    label_file_list:
      - ./train_data_det/val_list.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - DetLabelEncode: null
    - DetResizeForTest: null
    - NormalizeImage:
        scale: 1./255.
        mean:
        - 0.485
        - 0.456
        - 0.406
        std:
        - 0.229
        - 0.224
        - 0.225
        order: hwc
    - ToCHWImage: null
    - KeepKeys:
        keep_keys:
        - image
        - shape
        - polys
        - ignore_tags
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 1
    num_workers: 2
profiler_options: null
'''
with open('configs/det/PP-OCRv6/PP-OCRv6_small_det_finetune_colab.yml', 'w') as f:
    f.write(config_yaml)
print('config written')

In [ ]:
# 5. 학습 실행
!python tools/train.py -c configs/det/PP-OCRv6/PP-OCRv6_small_det_finetune_colab.yml

In [ ]:
# 6. (학습 끝난 후) 결과물을 구글드라이브로 백업
from google.colab import drive
drive.mount('/content/drive')
!cp -r output /content/drive/MyDrive/PP-OCRv6_small_det_finetune_output
print('backed up to Google Drive')